# Notebook 07 · CPS 主循环：两张耦合张量 W^C / W^S

L0–L3 是"一级加一种物理"的台阶，最终要装到**同一个级联循环**里跑，这就是 `cps_cascade.py`。

| 层 | 现实角色 | 跨层吃谁的东西 | 用哪张张量 |
|---|---|---|---|
| A 层 | 物理层（电网设备 / 生产线） | 信息层的指令 | `W^C = w⁰·exp(−λτ·τ̂)·h^α` |
| B 层 | 信息层（通信 / 调度系统） | 物理层的状态 | `W^S = w⁰·(1−β(1−h))·(1−q)` |

每轮顺序：**动力学迭代到不动点（每轮重算 W(t)）→ 判失效（δ 相对偏离 / φ 绝对底线 / Φ 感知满足度）→ 布尔耦合 → 超边逻辑 → GCC 剪枝**。

三条要记住的事：
1. `λτ = 0`、`β = 0` 时 W 退化成常数 `w⁰` —— 这就是 L1/L2 的"常数耦合"世界。
2. `δ = 1`、`φ = 0`、`Φ = 0` 时三级判据都不触发 ⇒ 必须**完全退化回 L0**（本 notebook 的 assert 格）。
3. 三个判据的分工：`δ` 是"相对自己健康态偏了多少"，`φ` 是"状态绝对底线"，`Φ` 是"**感知依赖满足度**"——`Φ` 是唯一能做到"**没有任何节点死亡也能判失效**"的口径（第 5 节）。


## 环境准备（每个 notebook 都要先跑这两格）


In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)


In [ ]:
"""准备一对 BA 网络 + 一批 k-of-N 功能单元，构造 CPS 主循环网络。"""
from phase6_interdependent.core import InterdependentNetwork
from phase6_interdependent.L3_hypergraph_logic import (
    HyperedgeNetwork, random_functional_hyperedges,
)
from phase6_interdependent import cps_cascade as CPS

G_A = nx.barabasi_albert_graph(200, 3, seed=42)   # A 层 = 物理层
G_B = nx.barabasi_albert_graph(200, 3, seed=43)   # B 层 = 信息层
print("双层网络就绪：N=%d / %d" % (G_A.number_of_nodes(), G_B.number_of_nodes()))

template = HyperedgeNetwork(G_A, G_B, seed=42)
units = random_functional_hyperedges(template, n_edges=40, size=3, rule="k-of-N", k=2, seed=42)
net0 = CPS.CPSNetwork(G_A, G_B, hyperedges=units, seed=42)
print("功能单元（超边）：", len(units), "条；F(未受损) =", net0.functional_measure())


## 1. W^C / W^S 长什么样


In [ ]:
"""W^C / W^S 长什么样（设计稿 §5.2 的形式）。

W^C_{iν} = w0 · exp(-λτ·τ̂_ν) · h_i^α      ← 设备 i 受控制器 ν 的指令
W^S_{νi} = w0 · (1 - β(1-h_i)) · (1 - q_ν) ← 传感器 ν 看设备 i 的状态
"""
print("—— W^C：指令有效性（设备健康度 h_i × 信息层时延 τ̂）——")
print("h_i   τ̂     0.0     0.3     0.6     1.0")
for h in (1.0, 0.8, 0.5, 0.2, 0.0):
    row = [CPS.control_weight(h, tau, lambda_tau=1.0, alpha=1.0) for tau in (0.0, 0.3, 0.6, 1.0)]
    print("%.1f        %.3f   %.3f   %.3f   %.3f" % (h, *row))

print("—— W^S：感知质量（设备健康度 h_i × 传感器丢包率 q）——")
print("h_i   q      0.0     0.3     0.6")
for h in (1.0, 0.8, 0.5, 0.2):
    row = [CPS.sensing_weight(h, q, beta=0.5) for q in (0.0, 0.3, 0.6)]
    print("%.1f        %.3f   %.3f   %.3f" % (h, *row))

print("读法：健康（h=1、τ̂=0、q=0）时两张张量都恰好是 1 ⇒ 与 L0 的常数耦合完全一样；")
print("      设备一退化、或时延一变大，跨层那份支撑就开始打折 —— 这就是 W(t) 连续化。")


## 2. 健康态参考与一次小攻击


In [ ]:
"""健康态参考：delta 判据比的是「相对自己没受损时的稳态差多少」。"""
cfg = CPS.CPSConfig(lambda_tau=1.0, alpha=1.0, beta=0.5, delta=0.3, phi=0.0)
x_ref = CPS.healthy_reference(G_A, G_B, cfg, seed=42)
for layer in ("A", "B"):
    vals = np.array(list(x_ref[layer].values()))
    print("%s 层健康态 x_ref：mean=%.3f  min=%.3f  max=%.3f" % (layer, vals.mean(), vals.min(), vals.max()))

probe = CPS.CPSNetwork(G_A, G_B, hyperedges=units, seed=42)
probe.run(sorted(G_A.nodes())[:20], rule=probe.cps_rule(cfg, x_ref))
print("受损 10%% 后：S(A)=%.3f  F=%.3f" % (probe.S("A"), probe.functional_measure()))


## 3. 四个旋钮逐个拧开


In [ ]:
"""四个旋钮逐个拧开：纯结构 -> 加容忍带 δ -> 加感知满足度 Φ -> 再把状态折扣（λτ、β）打开。"""
fractions = np.linspace(0.0, 0.4, 9)
settings = {
    "基线（λτ=0, β=0, δ=1, φs=0）": CPS.CPSConfig(lambda_tau=0.0, beta=0.0, delta=1.0),
    "只加容忍带 δ=0.3": CPS.CPSConfig(lambda_tau=0.0, beta=0.0, delta=0.3),
    "只加满足度 Φ=0.75": CPS.CPSConfig(lambda_tau=0.0, beta=0.0, delta=1.0, phi_s=0.75),
    "全开 λτ=1 + β=0.5 + δ=0.3 + Φ=0.75": CPS.CPSConfig(lambda_tau=1.0, beta=0.5, delta=0.3, phi_s=0.75),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, cfg in settings.items():
    net = CPS.CPSNetwork(G_A, G_B, hyperedges=units, seed=42)
    S, F = net.run_experiment_curves(cfg, fractions, n_trials=1, seed=42)
    axes[0].plot(fractions, S, "o-", markersize=4, label=label)
    axes[1].plot(fractions, F, "s-", markersize=4, label=label)
    print("%-34s S(0.2)=%.3f  F(0.2)=%.3f" % (label, np.interp(0.2, fractions, S), np.interp(0.2, fractions, F)))

for ax, ylabel, title in zip(axes, ("结构序参量 S", "功能度量 F"), ("结构视角", "功能视角")):
    ax.set_xlabel("初始移除比例 f")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("CPS 主循环：W^C / W^S 与双判据的作用")
fig.savefig(RESULTS / "notebook_07_cps_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("图已存：", RESULTS / "notebook_07_cps_curves.png")


## 5. 退化性检验（关掉三个判据 ⇒ 必须等于 L0）


## 4. 新失效通道：不删节点，只靠耦合劣化


In [ ]:
"""设计稿 §5.2 的**新失效通道**：不删任何节点，只让设备慢慢退化（x⁰_A 下降）。

δ、φ 全关掉，只留 Φ 口径 ⇒ 任何失效都只能来自"感知依赖满足不了"。
静态耦合模型里没有这条通道：静态的 Φ 只随"依赖节点是否存活"阶跃变化，
动态的 Φ 随 h 连续下降 ⇒ **连续劣化自己就能把系统推下悬崖**。
"""
print("h0     退化后 Φ_min   失效设备      S        F")
for h0 in (1.0, 0.8, 0.6, 0.5, 0.4, 0.2):
    cfg = CPS.CPSConfig(beta=0.5, delta=1.0, phi=0.0, phi_s=0.8, target_A=h0)
    net = CPS.CPSNetwork(G_A, G_B, hyperedges=units, seed=42)
    net.run([], rule=None)                        # 无人为故障
    net.run([], rule=net.cps_rule(cfg, None))     # 只靠 x⁰_A 下降驱动
    dead = sum(1 for n in net.G["A"] if not net.alive["A"][n])
    phi = min(CPS.sensing_satisfaction(net, cfg).values())
    print("%.1f    %.3f          %5d     %.3f    %.3f"
          % (h0, phi, dead, net.S("A"), net.functional_measure()))
print("\n读法：h0 ≥ 0.5 时零失效（Φ 还在 0.8 以上）；h0 ≤ 0.4 时整层被拖垮 ——")
print("      而**一个节点都没被人为删掉**。这条通道静态耦合模型里根本没有。")

In [ ]:
"""退化性检验：delta=1（关容忍）且 phi=0（关满足度）时，主循环必须退化成 L0。"""
from phase6_interdependent.L0_static_dependency import percolation_curve

# 注意：退化性检验要用**二元 AND 超边**（= 层间成对耦合，CPSNetwork 的默认值）。
# 本 notebook 上面的 units 是 k-of-N 功能单元，那是 L3 级别的物理，不参与 L0 退化对拍。
cfg0 = CPS.CPSConfig(lambda_tau=0.0, beta=0.0, delta=1.0, phi=0.0, phi_s=0.0)
fractions_small = np.array([0.0, 0.1, 0.2, 0.3])

def curve_cps():
    rng = np.random.default_rng(7)
    nodes = sorted(G_A.nodes())
    out = []
    for f in fractions_small:
        n_remove = int(round(len(nodes) * f))
        vals = []
        for _ in range(2):
            net = CPS.CPSNetwork(G_A, G_B, seed=7)   # 默认 = 二元 AND 超边（等价成对耦合）
            failed = [] if n_remove == 0 else rng.choice(nodes, size=n_remove, replace=False)
            net.run(failed, rule=net.cps_rule(cfg0))
            vals.append(net.S("A"))
        out.append(float(np.mean(vals)))
    return np.array(out)

rng = np.random.default_rng(7)
nodes = sorted(G_A.nodes())
s_plain = []
for f in fractions_small:
    n_remove = int(round(len(nodes) * f))
    vals = []
    for _ in range(2):
        net = InterdependentNetwork(G_A, G_B, seed=7)
        failed = [] if n_remove == 0 else rng.choice(nodes, size=n_remove, replace=False)
        vals.append(net.run(failed, rule=None)["S_A"])
    s_plain.append(float(np.mean(vals)))

s_cps = curve_cps()
print("CPS（关判据）:", np.round(s_cps, 6))
print("L0 基线      :", np.round(np.array(s_plain), 6))
assert np.allclose(s_cps, np.array(s_plain), atol=1e-9), "退化性检验失败：关掉判据后主循环必须等于 L0"
print("退化性检验通过：δ=1、φ=0、Φ=0 且 τ̂=q=0 时 CPS 主循环 == L0")


## 动手改一改（建议都试一遍）

1. 把 `β=0.5` 改成 `β=0`，其余不动：`S(f)` 与 `F(f)` 会怎么变？为什么在"只删节点"的攻击下差别很小？
2. 把 `φ` 设成 0.95：f=0（不攻击）时会不会就已经有节点不合格？这说明 `φ` 是"服务标准"而不是"攻击强度"。
3. 把超边规则从 `k-of-N` 换成 `AND`：同样的攻击下 F 掉得快多少？（对照 `06` 的 L3 notebook）
4. 打开 `src/phase6_interdependent/cps_cascade.py` 顶部教学块，对照本节每一格的输出读一遍。
